# 04 · Densidade, modelos de mistura e redução de dimensionalidade

**CRISP-DM fase 4** · DBSCAN, Mean Shift, GMM, PCA, Kernel PCA

Três famílias em que o número de grupos **emerge** do parâmetro, em vez de ser
imposto — e duas técnicas de projeção. Duas dessas famílias vão falhar neste conjunto,
e o diagnóstico da falha é o conteúdo desta etapa.

In [ ]:
import sys, warnings
from pathlib import Path

# O notebook roda a partir de `notebooks/`; a raiz do projeto entra no path
# para que `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from src import config
from src.utils.io import configurar_estilo

configurar_estilo()
print(f"Projeto em {RAIZ}")

In [ ]:
from src.data import dicionario as dic
from src.data.preparacao import carregar_curada, construir_matriz_modelagem, preparar_espaco
from src.model import densidade as den
from src.model import mistura as mis
from src.model import particional as part
from src.model import personas as pers
from src.model import reducao as red
from sklearn.cluster import KMeans

curada = carregar_curada()
matriz = construir_matriz_modelagem(curada, salvar=False)
espaco, nomes = preparar_espaco(matriz, "padrao")
K = 4
rotulos = pers.ordenar_canonicamente(
    curada, KMeans(K, n_init=20, random_state=config.SEMENTE).fit_predict(espaco)
)

## DBSCAN — calibrar `eps` pelo gráfico k-distância

Escolher `eps` por tentativa e erro é o erro mais comum do DBSCAN. O método correto:
ordenar a distância de cada ponto ao seu *k*-ésimo vizinho e procurar o joelho da
curva. `minPts` segue a heurística `2 × d`.

In [ ]:
min_amostras = 2 * espaco.shape[1]
curva = den.curva_k_distancia(espaco, min_amostras)
indice, eps = den.joelho_k_distancia(curva)
print(f"minPts = {min_amostras} · joelho em eps ≈ {eps:.3f}")

fig, eixo = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
eixo.plot(curva, color=config.AZUL_PROFUNDO, lw=2)
eixo.axhline(eps, ls="--", color=config.CORAL)
eixo.set_xlabel("registros ordenados")
eixo.set_ylabel(f"distância ao {min_amostras}º vizinho")
eixo.set_title("Gráfico k-distância")
plt.show()

In [ ]:
grade = np.round(np.arange(max(0.5, eps - 1.5), eps + 1.6, 0.25), 3)
den.varrer_dbscan(espaco, grade, min_amostras)

### O diagnóstico

Em toda a faixa de `eps`, o resultado é o mesmo: **um grupo gigante com mais de 90%
dos imóveis, mais ruído**. Não há vale de densidade neste espaço.

O motivo é geométrico. Em 21 dimensões padronizadas, os pontos se distribuem de forma
aproximadamente uniforme em densidade — não existem regiões densas separadas por
regiões vazias, que é a estrutura que o DBSCAN procura.

**Hipótese derrubada.** Mas o algoritmo não foi inútil: o que ele entrega de valor é o
**ruído**, e o ruído aqui é um produto — a lista de imóveis que não pertencem a
nenhuma linha e precisam de avaliação individual.

In [ ]:
rotulos_dbscan = den.aplicar_dbscan(espaco, 4.5, min_amostras)
ruido = rotulos_dbscan == -1
print(f"{ruido.sum()} imóveis fora de padrão ({100 * ruido.mean():.2f}% da carteira)")
den.perfil_do_ruido(curada, rotulos_dbscan, dic.NUMERICAS + dic.ORDINAIS)

In [ ]:
curada.loc[
    ruido,
    ["pid", "bairro", "area_habitavel", "qualidade_geral", "idade_imovel",
     "tipo_construcao", "preco_venda"],
].head(15)

## Mean Shift — o número de grupos emerge da largura de banda

In [ ]:
den.varrer_mean_shift(espaco)

Mesmo diagnóstico, pelo mesmo motivo: sem modos de densidade separados, a
busca converge quase toda para o mesmo modo. Repare que a silhueta **sobe** conforme o
resultado degenera — mais uma vez, métrica alta com resultado inútil.

## GMM — atribuição probabilística

O que o GMM traz e o K-Means não tem: **grau de pertencimento**. Um imóvel com 0,52 de
probabilidade no segmento A e 0,46 no B não é um imóvel do segmento A — é um imóvel de
fronteira.

A seleção de *k* usa BIC e AIC, e não a silhueta: como o GMM é probabilístico, existe
verossimilhança, e critérios de informação penalizam a complexidade explicitamente.

In [ ]:
varredura_gmm = mis.varrer_gmm(espaco, 2, 10)
varredura_gmm.pivot(index="k", columns="covariancia", values="bic").round(0)

### O que fazer quando o BIC não tem mínimo

Nenhuma das curvas apresenta mínimo no intervalo. Isso **não** é erro de execução: é o
que acontece quando os dados não são uma mistura de gaussianas bem separadas. Cada
componente a mais sempre melhora a verossimilhança o suficiente para vencer a
penalização.

O que se faz então: usar o **joelho da curva** e o teto operacional, e declarar a
limitação em vez de fingir que o BIC decidiu.

In [ ]:
bic_diag = varredura_gmm[varredura_gmm["covariancia"] == "diag"].set_index("k")["bic"]
print(f"Joelho da curva de BIC (covariância diag): k = {part.cotovelo(bic_diag)}")

melhor = varredura_gmm[varredura_gmm["k"] == K].sort_values("bic").iloc[0]["covariancia"]
print(f"Melhor covariância em k={K}: '{melhor}'")

In [ ]:
modelo_gmm, rotulos_gmm, probabilidades = mis.ajustar_gmm(espaco, K, melhor)
mis.classificar_confianca(probabilidades).value_counts()

In [ ]:
fig, eixo = plt.subplots(figsize=(9, 4), constrained_layout=True)
eixo.hist(probabilidades.max(axis=1), bins=40, color=config.ROXO, edgecolor="white")
eixo.axvline(0.60, ls="--", color=config.CORAL, label="fronteira")
eixo.axvline(0.80, ls="--", color=config.VERDE, label="núcleo")
eixo.set_xlabel("probabilidade do componente mais provável")
eixo.set_title("Grau de pertencimento")
eixo.legend()
plt.show()

## PCA — variância explicada e cargas

**Pré-requisito, não refinamento:** PCA sobre dados não padronizados encontra a
variável com maior unidade de medida, não a direção de maior informação. O espaço aqui
já está padronizado.

In [ ]:
modelo_pca, projecao = red.ajustar_pca(espaco)
variancia = red.variancia_explicada(modelo_pca)
print(f"Componentes para 80% da variância: {red.componentes_para(modelo_pca, 0.80)}")
variancia.head(12)

### As cargas — o que dá nome de negócio aos eixos

Sem esta tabela, "CP1" e "CP2" são rótulos vazios. Com ela, viram **porte e padrão** e
**maturidade do imóvel** — e a projeção 2D passa a ser defensável em uma banca.

In [ ]:
red.cargas(modelo_pca, [dic.rotulo(n) for n in nomes], quantos=3)

In [ ]:
fig, eixo = plt.subplots(figsize=(8, 6), constrained_layout=True)
for grupo in range(K):
    marcador = rotulos == grupo
    eixo.scatter(projecao[marcador, 0], projecao[marcador, 1], s=10, alpha=0.6,
                 color=pers.cores()[grupo], label=pers.codigo(grupo), edgecolors="none")
eixo.set_xlabel(f"CP1 — porte e padrão ({modelo_pca.explained_variance_ratio_[0]:.1%})")
eixo.set_ylabel(f"CP2 — maturidade ({modelo_pca.explained_variance_ratio_[1]:.1%})")
eixo.legend(markerscale=2)
eixo.set_title("Segmentos projetados nos dois primeiros componentes")
plt.show()

## Kernel PCA — legibilidade sem interpretabilidade

O Kernel PCA captura estrutura não linear e costuma produzir uma projeção mais
separada visualmente. O preço é a interpretabilidade: **não existem cargas**, porque a
projeção acontece em um espaço de dimensão implícita.

A tabela abaixo mede a silhueta **na projeção 2D**. Ela mede legibilidade do gráfico,
não qualidade do agrupamento — confundir as duas coisas é o erro que esta nota existe
para evitar.

In [ ]:
red.comparar_kernels(espaco, rotulos)

## LDA — por que ele não aparece aqui

`LinearDiscriminantAnalysis` é **supervisionado**: procura as direções que separam
classes conhecidas. Usá-lo antes de clusterizar seria contrabandear o rótulo reservado
para dentro do modelo — critério eliminatório da disciplina.

O LDA aparece uma única vez neste projeto, no notebook 06, sobre a faixa de preço
derivada de `preco_venda`, **depois** que os segmentos já estão definidos. É avaliação,
não modelagem.

## Matriz comparativa

Todas as métricas medidas no **mesmo espaço de referência**, independentemente do
espaço em que cada algoritmo foi ajustado. É a única forma de comparar.

In [ ]:
from src.pipeline.modelagem import matriz_comparativa
from src.model import distancias as dist
from src.model import hierarquica as hier
from sklearn.cluster import MeanShift, estimate_bandwidth

gower = dist.gower(matriz, dic.NUMERICAS + dic.ORDINAIS, dic.NOMINAIS)
largura = estimate_bandwidth(espaco, quantile=0.2, n_samples=1000, random_state=config.SEMENTE)

particoes = {
    "K-Means (euclidiano)": rotulos,
    "K-Means (euclid. + one-hot)": pers.ordenar_canonicamente(
        curada,
        KMeans(K, n_init=20, random_state=config.SEMENTE).fit_predict(
            preparar_espaco(matriz, "padrao", com_nominais=True)[0]
        ),
    ),
    "K-Medoids (Gower)": pers.ordenar_canonicamente(
        curada, part.kmedoids(gower, K, reinicios=12).rotulos
    ),
    "Hierárquica (Ward)": pers.ordenar_canonicamente(
        curada, hier.cortar(hier.matriz_ligacao(espaco, "ward"), K)
    ),
    "DBSCAN": rotulos_dbscan,
    "Mean Shift": MeanShift(bandwidth=largura, bin_seeding=True).fit_predict(espaco),
    "GMM": pers.ordenar_canonicamente(curada, rotulos_gmm),
}
comparativa = matriz_comparativa(espaco, particoes, gower, K)
comparativa[["grupos", "silhueta", "davies_bouldin", "pct_ruido", "maior_grupo_pct"]]

In [ ]:
comparativa[["forma_suposta", "representante", "leitura_de_negocio"]]

## Decisão

**K-Means sobre o espaço euclidiano padronizado, com k = 4.**

Não é o de maior silhueta bruta — DBSCAN e Mean Shift têm métricas melhores, e
degeneram. É o único que combina: partição equilibrada, estabilidade sob reamostragem,
quatro grupos interpretáveis em linguagem de negócio e aderência ao teto orçamentário
de cinco ações.